Globbing patterns can contain special characters which act as wildcards, such as:
- Asterisks (*), which match zero or more characters in a String. For example, the pattern *.csv `will` match any filename that ends in .csv.
- Question marks (?), `which match exactly one character`. For example, the pattern file?.csv will match files like file1.csv or fileA.csv but not file12.csv.
- Square brackets ([]),

In [25]:
import polars as pl
import polars.selectors as cs

df = pl.read_csv(
    "../data/raw/source=gps/date=2026-05-23/gps_*.csv"
    , has_header=False
#   , null_values="NA"
    , new_columns=
    [
        "transport_type"  # 1=trolleybus, 2=bus, 3=tram, 7=night bus
        , "line_number"  # string: e.g. `"8"`, `"18A"`
        , "longitude_raw"  # Longitude × 1,000,000 (WGS84)
        , "latitude_raw"  # Latitude × 1,000,000 (WGS84)
        , "speed_kmh"  # Empty string when unavailable
        , "heading_deg"  # `999` when unavailable
        , "vehicle_id"  # Internal vehicle identifier
        , "floor_type"  # Z = low-floor, false = unknown
        , "fleet_number"  # Physical vehicle serial number
        , "destination"  # Destination stop name
    ])

In [26]:
df.shape

(84762, 10)

In [27]:
df.glimpse(max_items_per_column=3)

Rows: 84762
Columns: 10
$ transport_type <i64> 3, 3, 3
$ line_number    <str> '2', '2', '1'
$ longitude_raw  <i64> 24668960, 24784690, 24706050
$ latitude_raw   <i64> 59461280, 59430110, 59452360
$ speed_kmh      <str> null, null, null
$ heading_deg    <i64> 315, 286, 320
$ vehicle_id     <i64> 98, 99, 102
$ floor_type     <str> 'Z', 'Z', 'Z'
$ fleet_number   <i64> 106, 144, 166
$ destination    <str> 'Kopli', 'Kopli', 'Kopli'



In [10]:
df.head()

transport_type,line_number,longitude_raw,latitude_raw,speed_kmh,heading_deg,vehicle_id,floor_type,fleet_number,destination
i64,str,i64,i64,str,i64,i64,str,i64,str
3,"""2""",24668960,59461280,null,315,98,"""Z""",106,"""Kopli"""
3,"""2""",24784690,59430110,null,286,99,"""Z""",144,"""Kopli"""
3,"""1""",24706050,59452360,null,320,102,"""Z""",166,"""Kopli"""
3,"""1""",24738840,59441090,null,89,110,"""Z""",31,"""Kadriorg"""
3,"""5""",24753360,59437210,null,57,123,"""Z""",64,"""Kopli"""


In [11]:
df.null_count().transpose(
    include_header=True, column_names=["null_count"]
)

column,null_count
str,u32
"""transport_type""",0
"""line_number""",0
"""longitude_raw""",0
"""latitude_raw""",0
"""speed_kmh""",84762
"""heading_deg""",0
"""vehicle_id""",0
"""floor_type""",0
"""fleet_number""",0


In [12]:
df.select("destination").unique()

destination
str
"""Kopli liinid"""
"""Harkujärve"""
"""Viimsi keskus"""
"""Mäeküla"""
"""Vesse"""
…
"""Liikuri"""
"""Väike-Õismäe"""
"""Lennujaam"""


In [13]:
df.select("speed_kmh").unique()

speed_kmh
str
null


In [14]:
df.select([cs.all(), pl.col("transport_type").lt(3).alias("lt3")])

transport_type,line_number,longitude_raw,latitude_raw,speed_kmh,heading_deg,vehicle_id,floor_type,fleet_number,destination,lt3
i64,str,i64,i64,str,i64,i64,str,i64,str,bool
3,"""2""",24668960,59461280,null,315,98,"""Z""",106,"""Kopli""",false
3,"""2""",24784690,59430110,null,286,99,"""Z""",144,"""Kopli""",false
3,"""1""",24706050,59452360,null,320,102,"""Z""",166,"""Kopli""",false
3,"""1""",24738840,59441090,null,89,110,"""Z""",31,"""Kadriorg""",false
3,"""5""",24753360,59437210,null,57,123,"""Z""",64,"""Kopli""",false
…,…,…,…,…,…,…,…,…,…,…
2,"""31""",24896140,59447340,null,133,3781,"""Z""",6,"""Priisle""",true
2,"""63""",24791360,59442570,null,257,3790,"""Z""",75,"""Maneezi""",true
2,"""50""",24896790,59445170,null,16,3794,"""Z""",76,"""Seli""",true


In [15]:
df_alias = df.with_columns(pl
             .when(pl.col.transport_type == 1).then(pl.lit("Trolleybus"))
             .when(pl.col.transport_type == 2).then(pl.lit("Bus"))
             .when(pl.col.transport_type == 3).then(pl.lit("Tram"))
             .when(pl.col.transport_type == 7).then(pl.lit("Night bus"))
             .otherwise(pl.lit("Unknown"))
             .alias("type")
             )
df_alias

transport_type,line_number,longitude_raw,latitude_raw,speed_kmh,heading_deg,vehicle_id,floor_type,fleet_number,destination,type
i64,str,i64,i64,str,i64,i64,str,i64,str,str
3,"""2""",24668960,59461280,null,315,98,"""Z""",106,"""Kopli""","""Tram"""
3,"""2""",24784690,59430110,null,286,99,"""Z""",144,"""Kopli""","""Tram"""
3,"""1""",24706050,59452360,null,320,102,"""Z""",166,"""Kopli""","""Tram"""
3,"""1""",24738840,59441090,null,89,110,"""Z""",31,"""Kadriorg""","""Tram"""
3,"""5""",24753360,59437210,null,57,123,"""Z""",64,"""Kopli""","""Tram"""
…,…,…,…,…,…,…,…,…,…,…
2,"""31""",24896140,59447340,null,133,3781,"""Z""",6,"""Priisle""","""Bus"""
2,"""63""",24791360,59442570,null,257,3790,"""Z""",75,"""Maneezi""","""Bus"""
2,"""50""",24896790,59445170,null,16,3794,"""Z""",76,"""Seli""","""Bus"""


In [16]:
df_alias.select('type').unique()

type
str
"""Bus"""
"""Tram"""


In [17]:
df_alias.select('transport_type', 'type', 'line_number')

transport_type,type,line_number
i64,str,str
3,"""Tram""","""2"""
3,"""Tram""","""2"""
3,"""Tram""","""1"""
3,"""Tram""","""1"""
3,"""Tram""","""5"""
…,…,…
2,"""Bus""","""31"""
2,"""Bus""","""63"""
2,"""Bus""","""50"""


In [23]:
buss33 = df_alias.filter(pl.col('line_number') == "33")

In [24]:
buss33.is_duplicated().sum()

217